In [1]:
!pip install psycopg2-binary pandas lxml

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ----------- ---------------------------- 0.8/2.7 MB 4.2 MB/s eta 0:00:01
   --------------------------- ------------ 1.8/2.7 MB 4.2 MB/s eta 0:00:01
   ---------------------------------------- 2.7/2.7 MB 4.1 MB/s  0:00:00



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


##### Create Connection

In [2]:
import psycopg2

def get_connection():
    conn = psycopg2.connect(
        host="localhost",
        database="kenexaihackathon",
        user="postgres",
        password="09092002",
        port="5432"
    )
    return conn

### Auvik Payload

In [23]:
import json

file_path = "../../data/raw/Auvik Payload.json"

conn = get_connection()
cur = conn.cursor()

with open(file_path, "r") as f:
    alerts = json.load(f)

for alert in alerts:
    cur.execute("""
        INSERT INTO bronze.auvik_alerts_raw (
            entity_id,
            subject,
            alert_status_string,
            alert_id,
            alert_name,
            entity_name,
            company_name,
            entity_type,
            alert_date,
            link,
            alert_status,
            correlation_id,
            alert_description,
            alert_severity_string,
            alert_severity,
            company_id
        )
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """, (
        alert.get("entityId"),
        alert.get("subject"),
        alert.get("alertStatusString"),
        alert.get("alertId"),
        alert.get("alertName"),
        alert.get("entityName"),
        alert.get("companyName"),
        alert.get("entityType"),
        alert.get("date"),
        alert.get("link"),
        alert.get("alertStatus"),
        alert.get("correlationId"),
        alert.get("alertDescription"),
        alert.get("alertSeverityString"),
        alert.get("alertSeverity"),
        alert.get("companyId")
    ))

conn.commit()
cur.close()
conn.close()

print("Auvik ingestion completed")


Auvik ingestion completed


### Meraki Payload

In [16]:
import json

file_path = "../../data/raw/Meraki Payload.json"

conn = get_connection()
cur = conn.cursor()

with open(file_path, "r") as f:
    alerts = json.load(f)

for alert in alerts:

    cur.execute("""
        INSERT INTO bronze.meraki_alerts_raw (
            app_key,
            alert_status,
            check_name,
            payload_version,
            shared_secret,
            sent_at,
            organization_id,
            organization_name,
            organization_url,
            network_id,
            network_name,
            network_url,
            device_serial,
            device_mac,
            device_name,
            device_url,
            device_model,
            alert_id,
            alert_type,
            alert_type_id,
            alert_level,
            occurred_at,
            device_host
        )
        VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
    """, (
        alert.get("app_key"),
        alert.get("status"),
        alert.get("check"),
        alert.get("version"),
        alert.get("sharedSecret"),
        alert.get("sentAt"),
        alert.get("organizationId"),
        alert.get("organizationName"),
        alert.get("organizationUrl"),
        alert.get("networkId"),
        alert.get("networkName"),
        alert.get("networkUrl"),
        alert.get("deviceSerial"),
        alert.get("deviceMac"),
        alert.get("deviceName"),
        alert.get("deviceUrl"),
        alert.get("deviceModel"),
        alert.get("alertId"),
        alert.get("alertType"),
        alert.get("alertTypeId"),
        alert.get("alertLevel"),
        alert.get("occurredAt"),
        alert.get("host")
    ))

conn.commit()
cur.close()
conn.close()

print("Meraki ingestion completed")

Meraki ingestion completed


### N-Central


In [22]:
import xml.etree.ElementTree as ET

file_path = "../../data/raw/N-Central Payload.xml"

conn = get_connection()
cur = conn.cursor()

with open(file_path, "r", encoding="utf-8") as f:
    content = f.read()

# Split multiple XML declarations
xml_documents = [doc.strip() for doc in content.split('<?xml version="1.0" encoding="UTF-8"?>') if doc.strip()]

for xml_doc in xml_documents:
    full_xml = '<?xml version="1.0" encoding="UTF-8"?>' + xml_doc
    root = ET.fromstring(full_xml)

    cur.execute("""
    INSERT INTO bronze.ncentral_alerts_raw (
        active_notification_trigger_id,
        customer_name,
        device_uri,
        device_name,
        affected_service,
        task_ident,
        ncentral_uri,
        qualitative_old_state,
        qualitative_new_state,
        time_of_state_change,
        probe_uri,
        quantitative_new_state,
        service_organization_name,
        remote_control_link
    )
    VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
    """, (
        root.findtext("ActiveNotificationTriggerID"),
        root.findtext("CustomerName"),
        root.findtext("DeviceURI"),
        root.findtext("DeviceName"),
        root.findtext("AffectedService"),
        root.findtext("TaskIdent"),
        root.findtext("NcentralURI"),
        root.findtext("QualitativeOldState"),
        root.findtext("QualitativeNewState"),
        root.findtext("TimeOfStateChange"),
        root.findtext("ProbeURI"),
        root.findtext("QuantitativeNewState"),
        root.findtext("ServiceOrganizationName"),
        root.findtext("RemoteControlLink")
    ))

conn.commit()
cur.close()
conn.close()

print("NCentral ingestion completed")


NCentral ingestion completed
